In [55]:
import pandas as pd

df = pd.read_parquet('data.parquet')

In [56]:
print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")


Linhas: 20000 | Colunas: 27


In [57]:
df.isnull().sum()

Income                             0
Age                                0
Dependents                         0
Occupation                         0
City_Tier                          0
Rent                               0
Loan_Repayment                     0
Insurance                          0
Groceries                          0
Transport                          0
Eating_Out                         0
Entertainment                      0
Utilities                          0
Healthcare                         0
Education                          0
Miscellaneous                      0
Desired_Savings_Percentage         0
Desired_Savings                    0
Disposable_Income                  0
Potential_Savings_Groceries        0
Potential_Savings_Transport        0
Potential_Savings_Eating_Out       0
Potential_Savings_Entertainment    0
Potential_Savings_Utilities        0
Potential_Savings_Healthcare       0
Potential_Savings_Education        0
Potential_Savings_Miscellaneous    0
d

In [58]:
# Bloco 5
# Entendendo estatísticas das colunas
df[['Income', 'Disposable_Income', 'Desired_Savings', 'Loan_Repayment']].describe()

,Income,Disposable_Income,Desired_Savings,Loan_Repayment
count,2.000000e+04,20000.000000,20000.000000,20000.000000
mean,4.158550e+04,10647.367257,4982.878416,2049.800292
std,4.001454e+04,11740.637289,7733.468188,4281.789941
min,1.301187e+03,-5400.788673,0.000000,0.000000
25%,1.760488e+04,3774.894323,1224.932636,0.000000
50%,3.018538e+04,7224.890977,2155.356763,0.000000
75%,5.176545e+04,13331.950716,6216.309609,2627.142320
max,1.079728e+06,377060.218482,245504.485208,123080.682009


In [59]:
# Bloco 6 -- Investigando zeros em desired_savings

(df['Desired_Savings'] == 0).sum()

np.int64(112)

In [60]:
# Bloco 7 -- Removendo clientes sem meta de poupança
df = df[df['Desired_Savings'] > 0]

In [61]:
# Bloco 9 -- Entendendo os gastos não essenciais
# Coluna auxiliar com o percentual que cada cliente gasta em
# Entretenimento e alimentação fora, em relação a renda

df['perc_nao_essenciais'] = (df['Eating_Out'] + df['Entertainment']) / df['Income']

In [62]:
# Definindo a condição 1, o quão pesado é o empréstimo nos clientes que estao com emprestimos ativos
df['perc_emprestimo'] = df['Loan_Repayment'] / df['Income']
df['perc_emprestimo'].describe()

count    19888.000000
mean         0.049144
std          0.066538
min          0.000000
25%          0.000000
50%          0.000000
75%          0.104682
max          0.199938
Name: perc_emprestimo, dtype: float64

In [63]:
# Validando a dimensão 4 -- a ausência  de margem
# para emergências. Precisamos somar todas as colunas de 
# poupança potencial e ver quanto cada cliente poderia economizar
# mas não economiza, em relação a renda

colunas_potencial = [
    'Potential_Savings_Groceries',
    'Potential_Savings_Transport',
    'Potential_Savings_Eating_Out',
    'Potential_Savings_Entertainment',
    'Potential_Savings_Utilities',
    'Potential_Savings_Healthcare',
    'Potential_Savings_Education',
    'Potential_Savings_Miscellaneous'
]


df['total_potential_savings'] = df[colunas_potencial].sum(axis=1)
df['perc_potential_savings'] = df['total_potential_savings'] / df['Income']
df['perc_potential_savings'].describe

# Criando buffer_emergencia -- apos disponibilizar o valor disponivel para a poupança, o quanto sobre para emergencia

df['buffer_emergencia'] = (df['Disposable_Income'] - df['Desired_Savings']) / df['Income']
df['buffer_emergencia'].describe()

count    19888.000000
mean         0.161962
std          0.095972
min          0.000000
25%          0.087768
50%          0.164344
75%          0.235677
max          0.448696
Name: buffer_emergencia, dtype: float64

In [64]:
# Bloco 10 -- Criando o novo risk score

c1 = (df['perc_emprestimo'] > 0.10).astype(int)
c2 = (df['perc_nao_essenciais'] > 0.085).astype(int)
c3 = (df['buffer_emergencia'] < 0.10).astype(int)
c4 = (df['perc_potential_savings'] > 0.08).astype(int)

df['risk_score'] = c1 + c2 + c3 + c4
df['risk_score'].value_counts().sort_index()

risk_score
0    10291
1     5066
2     3690
3      763
4       78
Name: count, dtype: int64

In [65]:
# Definição do target para identificar a partir de quantas condições
# o usuário será definido como vulnerável

df['Vulnerable'] = (df['risk_score'] >= 2).astype(int)
df['Vulnerable'].value_counts()

Vulnerable
0    15357
1     4531
Name: count, dtype: int64

In [66]:
# Bloco 14 -- Comparando os dois grupos 

df.groupby('Vulnerable')[['Income', 'perc_emprestimo', 'perc_nao_essenciais', 'buffer_emergencia', 'perc_potential_savings']].mean().round(4)

,Income,perc_emprestimo,perc_nao_essenciais,buffer_emergencia,perc_potential_savings
Vulnerable,,,,,
0,39739.8395,0.0246,0.0688,0.1953,0.0610
1,47741.3189,0.1322,0.0739,0.0490,0.0655


In [67]:
# Bloco 15 -- Gráfico comparativo dos grupos

import matplotlib.pyplot as plt

grupos = df.groupby('Vulnerable')[['perc_emprestimo', 'perc_nao_essenciais', 'buffer_emergencia', 'perc_potential_savings']].mean()

grupos.T.plot(kind='bar', figsize=(10, 5), color=['#43a047', '#e53935'])
plt.title('Perfil Médio: Clientes Seguros vs Vulneráveis')
plt.ylabel('Valor médio')
plt.xticks(rotation=30)
plt.legend(['Seguro (0)', 'Vulnerável (1)'])
plt.tight_layout()
plt.savefig('comparativo_grupos.png', dpi=150)
plt.show()

/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/1679432167.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [68]:
#Bloco 16 -- Definindo features e target 
# Features -- As colunas que vão servir de aprendizado ao modelo

# criando feature savings_rates
df['savings_rate'] = df['Desired_Savings'] / df['Income']

features = ['Income', 'Age', 'Dependents', 'Loan_Repayment', 'Eating_Out', 
            'Entertainment', 'Healthcare', 'savings_rate', 'perc_emprestimo',
            'perc_nao_essenciais', 'buffer_emergencia', 'perc_potential_savings']

X = df[features]
y = df['Vulnerable']

In [69]:
# Bloco 17 -- Matriz de Correlação (Decidir a relação das features e quais remover)

import matplotlib.pyplot as plt
import seaborn as sns

# Matriz de correlação entre as features
corr_matrix = X.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    square=True,
    linewidths=0.5,
    annot_kws={"size": 8}
)
plt.title('Matriz de Correlação entre Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacao_features.png', dpi=150)
plt.show()

print("\nPares com correlação alta (> 0.7 ou < -0.7):")
for col in corr_matrix.columns:
    for row in corr_matrix.index:
        if col != row and abs(corr_matrix.loc[row, col]) > 0.7:
            print(f"  {row} ↔ {col}: {corr_matrix.loc[row, col]:.2f}")


Pares com correlação alta (> 0.7 ou < -0.7):
  Eating_Out ↔ Income: 0.94
  Entertainment ↔ Income: 0.94
  Healthcare ↔ Income: 0.98
  savings_rate ↔ Income: 0.71
  Income ↔ Eating_Out: 0.94
  Entertainment ↔ Eating_Out: 0.89
  Healthcare ↔ Eating_Out: 0.92
  Income ↔ Entertainment: 0.94
  Eating_Out ↔ Entertainment: 0.89
  Healthcare ↔ Entertainment: 0.92
  Income ↔ Healthcare: 0.98
  Eating_Out ↔ Healthcare: 0.92
  Entertainment ↔ Healthcare: 0.92
  Income ↔ savings_rate: 0.71


/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/1391561298.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Bloco 18 -- Redução da dimensionalidade, alterações das features
# Removendo Eating_Out, Entertainment, HealthCare, savings_rate e Loan_repayment
# Removendo features redundantes — substituídas pelas versões em %
features_reduzidas = [
    'Income', 'Age', 'Dependents', 'Loan_Repayment',
              'Eating_Out', 'Entertainment', 'Healthcare',
              'Rent', 'Groceries', 'Disposable_Income', 'Desired_Savings'
]



X = df[features_reduzidas]
y = df['Vulnerable']

print(f"Features anteriores: 12")
print(f"Features após redução: {X.shape[1]}")
print(f"\nFeatures mantidas: {features_reduzidas}")

Features anteriores: 12
Features após redução: 11

Features mantidas: ['Income', 'Age', 'Dependents', 'Loan_Repayment', 'Eating_Out', 'Entertainment', 'Healthcare', 'Rent', 'Groceries', 'Disposable_Income', 'Desired_Savings']


In [ ]:
# Bloco 19 -- Divisão treino e teste
# Evitar overfitting (decorar dados e replicar, não aprender)
# Separação dos dados em dois grupos: 80% treino e 20% teste

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino:  {X_train.shape[0]} clientes")
print(f"Teste:   {X_test.shape[0]} clientes")

Treino:  15910 clientes
Teste:   3978 clientes


In [72]:
# Bloco 20 -- Normalização dos dados para uma mesma escala

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Normalização concluída!")

Normalização concluída!


In [73]:
# Bloco 21 -- Treinando os 3 Modelos

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

lr = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
lr.fit(X_train_scaled, y_train)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train_scaled, y_train)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_scaled, y_train)

print("✅ Regressão Logística treinada")
print("✅ Random Forest treinada")
print("✅ Gradient Boosting treinado")

✅ Regressão Logística treinada
✅ Random Forest treinada
✅ Gradient Boosting treinado


In [74]:
# Bloco 22 - Avaliação dos Modelos

from sklearn.metrics import classification_report, roc_auc_score

modelos = {
    'Regressão Logística': lr,
    'Random Forest': rf,
    'Gradient Boosting': gb
}

resultados = {}

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test_scaled)
    y_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    resultados[nome] = {'auc': auc, 'y_pred': y_pred, 'y_prob': y_prob}
    print(f"\n{'='*50}")
    print(f"  {nome} — AUC-ROC: {auc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Seguro', 'Vulnerável']))


  Regressão Logística — AUC-ROC: 0.9505
              precision    recall  f1-score   support

      Seguro       0.97      0.90      0.93      3072
  Vulnerável       0.73      0.91      0.81       906

    accuracy                           0.90      3978
   macro avg       0.85      0.90      0.87      3978
weighted avg       0.92      0.90      0.91      3978


  Random Forest — AUC-ROC: 0.9724
              precision    recall  f1-score   support

      Seguro       0.94      0.98      0.96      3072
  Vulnerável       0.92      0.80      0.85       906

    accuracy                           0.94      3978
   macro avg       0.93      0.89      0.91      3978
weighted avg       0.94      0.94      0.94      3978


  Gradient Boosting — AUC-ROC: 0.9631
              precision    recall  f1-score   support

      Seguro       0.94      0.97      0.95      3072
  Vulnerável       0.88      0.78      0.83       906

    accuracy                           0.93      3978
   macro avg 

In [75]:
# Bloco 23 -- Matriz de Confusão do Random Forest
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

y_pred_rf = resultados['Random Forest']['y_pred']
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Seguro', 'Vulnerável'],
            yticklabels=['Seguro', 'Vulnerável'])
plt.title('Matriz de Confusão — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Real')
plt.xlabel('Previsto pelo Modelo')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nVerdadeiros Negativos (Seguro → Seguro):        {tn}")
print(f"Falsos Positivos    (Seguro → Vulnerável):       {fp}")
print(f"Falsos Negativos    (Vulnerável → Seguro):       {fn}")
print(f"Verdadeiros Positivos (Vulnerável → Vulnerável): {tp}")
print(f"\nDe {tp+fn} vulneráveis reais, o modelo identificou {tp} corretamente ({tp/(tp+fn)*100:.1f}%)")


Verdadeiros Negativos (Seguro → Seguro):        3011
Falsos Positivos    (Seguro → Vulnerável):       61
Falsos Negativos    (Vulnerável → Seguro):       184
Verdadeiros Positivos (Vulnerável → Vulnerável): 722

De 906 vulneráveis reais, o modelo identificou 722 corretamente (79.7%)


/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/2054286939.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [76]:
# Bloco 25 -- SHAP: Explicabilidade do Modelo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Redefinindo features_b
features_b = ['Income', 'Age', 'Dependents', 'Loan_Repayment',
              'Eating_Out', 'Entertainment', 'Healthcare',
              'Rent', 'Groceries', 'Disposable_Income', 'Desired_Savings']

# Shape (3978, 11, 2) → pegamos a classe 1 (Vulnerável)
sv = shap_values[:, :, 1]

# Calculando importância média
mean_shap = np.abs(sv).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': features_b,
    'importancia': mean_shap
}).sort_values('importancia', ascending=True)

print(shap_df)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(shap_df['feature'], shap_df['importancia'], color='#1976d2')
ax.set_title('Importância das Features — SHAP Values', fontweight='bold', fontsize=13)
ax.set_xlabel('Importância média (|SHAP value|)')
plt.tight_layout()
plt.savefig('shap_importancia.png', dpi=150)
plt.show()
print("✅ Gráfico SHAP salvo!")

              feature  importancia
1                 Age     0.006796
2          Dependents     0.010867
6          Healthcare     0.015745
8           Groceries     0.016264
10    Desired_Savings     0.018664
0              Income     0.021016
7                Rent     0.026901
4          Eating_Out     0.034798
5       Entertainment     0.036644
9   Disposable_Income     0.138478
3      Loan_Repayment     0.187915
✅ Gráfico SHAP salvo!


/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/1717808074.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [77]:
# Principio de Pareto 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

features_b = ['Income', 'Age', 'Dependents', 'Loan_Repayment',
              'Eating_Out', 'Entertainment', 'Healthcare',
              'Rent', 'Groceries', 'Disposable_Income', 'Desired_Savings']

sv = shap_values[:, :, 1]
mean_shap = np.abs(sv).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': features_b,
    'importancia': mean_shap
}).sort_values('importancia', ascending=False)

shap_df['importancia_pct'] = shap_df['importancia'] / shap_df['importancia'].sum() * 100
shap_df['cumulativo'] = shap_df['importancia_pct'].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(shap_df['feature'], shap_df['importancia_pct'], color='#1976d2', label='Importância %')
ax1.set_ylabel('Importância individual (%)', color='#1976d2')
ax1.set_xticklabels(shap_df['feature'], rotation=30, ha='right')
ax1.set_ylim(0, 50)

ax2 = ax1.twinx()
ax2.plot(shap_df['feature'], shap_df['cumulativo'], color='#e53935',
         marker='o', linewidth=2.5, label='Cumulativo %')
ax2.axhline(y=80, color='gray', linestyle='--', linewidth=1, label='80%')
ax2.set_ylabel('Importância cumulativa (%)', color='#e53935')
ax2.set_ylim(0, 110)

plt.title('Princípio de Pareto — Importância das Features', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.savefig('pareto_features.png', dpi=150)
plt.show()
print("✅ Gráfico de Pareto salvo!")

✅ Gráfico de Pareto salvo!


/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/3567279273.py:26: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_xticklabels(shap_df['feature'], rotation=30, ha='right')
/var/folders/wg/rtj_csj552q4bb7syb0n4ghw0000gn/T/ipykernel_49089/3567279273.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [78]:
# Bloco Final -- Segmentação por Risco e Impacto Financeiro

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

features_b = ['Income', 'Age', 'Dependents', 'Loan_Repayment',
              'Eating_Out', 'Entertainment', 'Healthcare',
              'Rent', 'Groceries', 'Disposable_Income', 'Desired_Savings']

# Rodando o modelo em toda a base
X_completo = df[features_b]
X_completo_scaled = scaler.transform(X_completo)
df['probabilidade_risco'] = rf.predict_proba(X_completo_scaled)[:, 1]

# Segmentando por nível de risco
def classificar_risco(prob):
    if prob < 0.30:
        return 'Baixo'
    elif prob < 0.60:
        return 'Médio'
    else:
        return 'Alto'

df['nivel_risco'] = df['probabilidade_risco'].apply(classificar_risco)

# Tabela de segmentação
segmentacao = df.groupby('nivel_risco').agg(
    clientes=('Income', 'count'),
    renda_media=('Income', 'mean'),
    emprestimo_medio=('Loan_Repayment', 'mean'),
    prob_media=('probabilidade_risco', 'mean')
).reindex(['Baixo', 'Médio', 'Alto'])

segmentacao['pct_base'] = (segmentacao['clientes'] / len(df) * 100).round(1)
segmentacao['exposicao_mensal'] = segmentacao['clientes'] * segmentacao['emprestimo_medio']

print("=" * 65)
print("  SEGMENTAÇÃO DE CLIENTES POR NÍVEL DE RISCO")
print("=" * 65)
for nivel in ['Baixo', 'Médio', 'Alto']:
    row = segmentacao.loc[nivel]
    print(f"\n🔹 Risco {nivel}:")
    print(f"   Clientes:            {int(row['clientes']):,}")
    print(f"   % da base:           {row['pct_base']}%")
    print(f"   Prob. média:         {row['prob_media']*100:.1f}%")
    print(f"   Empréstimo médio:    R$ {row['emprestimo_medio']:,.0f}")
    print(f"   Exposição mensal:    R$ {row['exposicao_mensal']:,.0f}")

# Impacto financeiro do grupo de alto risco
alto_risco = segmentacao.loc['Alto']
exposicao_total = alto_risco['exposicao_mensal']
economia_30pct = exposicao_total * 0.30

print("\n" + "=" * 65)
print("  IMPACTO FINANCEIRO — GRUPO DE ALTO RISCO")
print("=" * 65)
print(f"\n  Clientes em alto risco:     {int(alto_risco['clientes']):,}")
print(f"  Exposição mensal total:     R$ {exposicao_total:,.0f}")
print(f"  Economia potencial (30%):   R$ {economia_30pct:,.0f}")
print(f"\n  * Estimativa conservadora: prevenção de 30% das inadimplências")
print(f"    com ação antecipada (contato, renegociação, limite)")

  SEGMENTAÇÃO DE CLIENTES POR NÍVEL DE RISCO

🔹 Risco Baixo:
   Clientes:            15,271
   % da base:           76.8%
   Prob. média:         2.8%
   Empréstimo médio:    R$ 784
   Exposição mensal:    R$ 11,977,296

🔹 Risco Médio:
   Clientes:            309
   % da base:           1.6%
   Prob. média:         44.2%
   Empréstimo médio:    R$ 3,644
   Exposição mensal:    R$ 1,126,110

🔹 Risco Alto:
   Clientes:            4,308
   % da base:           21.7%
   Prob. média:         90.7%
   Empréstimo médio:    R$ 6,255
   Exposição mensal:    R$ 26,947,436

  IMPACTO FINANCEIRO — GRUPO DE ALTO RISCO

  Clientes em alto risco:     4,308
  Exposição mensal total:     R$ 26,947,436
  Economia potencial (30%):   R$ 8,084,231

  * Estimativa conservadora: prevenção de 30% das inadimplências
    com ação antecipada (contato, renegociação, limite)


In [79]:
# Implementar o Modelo para Utilização Posterior: 

import joblib

# Salvando o modelo e o scaler
joblib.dump(rf, 'modelo_vulnerabilidade.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("✅ Modelo salvo: modelo_vulnerabilidade.pkl")
print("✅ Scaler salvo: scaler.pkl")

✅ Modelo salvo: modelo_vulnerabilidade.pkl
✅ Scaler salvo: scaler.pkl


In [80]:
import os
print(os.getcwd())

/Users/jmccl/modelo_churn


In [81]:
# Salvando o dataset limpo que foi usado no treinamento
df_limpo = df[df['Disposable_Income'] >= 0].copy()
df_limpo.to_parquet('data_limpo.parquet', index=False)
print(f"✅ Dataset limpo salvo: {len(df_limpo):,} clientes")

✅ Dataset limpo salvo: 19,888 clientes
